# KLEOS 02 — QLoRA training**This notebook is the canonical way to train a KLEOS adapter.**It runs the real pipeline: 4-bit quantized base model, LoRA adapter, assistant-only loss masking, a genuine `Trainer` loop, checkpointing, and a full experimentmanifest.## Before you start1. **Set the runtime to GPU.** Runtime → Change runtime type → T4 GPU.2. **Expect to be disconnected.** Free Colab reclaims runtimes without warning.   Section 7 writes checkpoints to Drive so a disconnect costs minutes, not the   session.3. **Check feasibility first.** Section 5 tells you whether your chosen model fits   the GPU you were assigned.

## 1. Clone and install

In [ ]:
# Clone the repository (skip if already present) and enter it.import osfrom pathlib import PathREPO_DIR = Path("/content/kleos-models")if not REPO_DIR.exists():    !git clone https://github.com/kleos/kleos-models.git {REPO_DIR}else:    print(f"{REPO_DIR} already exists; pulling latest")    !cd {REPO_DIR} && git pull --ff-onlyos.chdir(REPO_DIR)print("working directory:", Path.cwd())

In [ ]:
# Install dependencies WITHOUT touching Colab's torch build.## Reinstalling torch on Colab replaces the build compiled against this runtime's# CUDA driver, and CUDA then silently stops working. scripts/colab_setup.py uses# --no-deps for every package that would otherwise pull torch along.!python scripts/colab_setup.py

## 2. Authenticate

In [ ]:
# Hugging Face authentication.## Needed only for gated base models (Mistral) or to publish an adapter.# Use Colab Secrets (the key icon in the left sidebar), never a literal token in# a cell — notebooks get shared and committed.import ostry:    from google.colab import userdata    token = userdata.get("HF_TOKEN")    if token:        os.environ["HF_TOKEN"] = token        print("HF_TOKEN loaded from Colab secrets.")    else:        print("No HF_TOKEN secret set.")except Exception as exc:    print(f"Colab secrets unavailable ({type(exc).__name__}).")    print("Set os.environ['HF_TOKEN'] manually if you need gated models.")if not os.environ.get("HF_TOKEN"):    print()    print("Without a token you can still use ungated models such as Qwen/Qwen3-8B.")    print("To add one: sidebar key icon -> Add new secret -> name HF_TOKEN ->")    print("enable 'Notebook access'.")

## 3. Check the GPU

In [ ]:
# What GPU did Colab actually assign? Free-tier allocation varies.!nvidia-smifrom kleos_models.models.feasibility import probe_gpugpu = probe_gpu()print()print(gpu.render())print()if not gpu.available:    print("NO GPU ASSIGNED.")    print("Runtime -> Change runtime type -> T4 GPU, then re-run this cell.")elif not gpu.bf16_supported:    print(f"Note: {gpu.name} (compute capability {gpu.capability_string}) has no")    print("bfloat16 support. KLEOS configs use compute_dtype: auto, which selects")    print("float16 here automatically. Nothing to change.")

## 4. Choose a configuration`configs/training/debug.yaml` runs ten steps and proves the pipeline works.`configs/training/qlora_small.yaml` is the real run.Model configs available:| Config | Fits a free T4? || --- | --- || `qwen3_8b` | yes (4-bit) || `ministral_8b` | yes (4-bit) — gated, needs HF_TOKEN || `mistral_small_3_2` | no — needs A100 || `qwen3_30b_a3b_thinking` | no — needs A100 |

In [ ]:
CONFIG = "configs/training/qlora_small.yaml"MODEL = "configs/models/qwen3_8b.yaml"DATASET = "data/examples"   # replace with your dataset directory# Start with the debug config to prove the pipeline runs end to end:# CONFIG = "configs/training/debug.yaml"print(f"config : {CONFIG}")print(f"model  : {MODEL}")print(f"dataset: {DATASET}")

## 5. Will it fit?Do this **before** downloading weights. If the answer is no, change the modelhere rather than discovering it during training.

In [ ]:
!python scripts/plan_run.py --config {CONFIG} --set model.name=check

If this reports `infeasible` or `inference_only`, switch to an 8B config. Thepipeline will refuse to start rather than silently shrink your configuration intosomething that is no longer comparable with other runs.

## 6. Validate the dataset

In [ ]:
!python scripts/validate_dataset.py --dataset {DATASET}

## 7. Persist checkpoints to Drive (strongly recommended)Colab runtimes disappear. Writing checkpoints to Drive means a disconnect costsyou the time since the last checkpoint, not the whole session.

In [ ]:
USE_DRIVE = True   # set False to keep checkpoints on the ephemeral runtime diskif USE_DRIVE:    from google.colab import drive    drive.mount("/content/drive")    OUTPUT_DIR = "/content/drive/MyDrive/kleos/outputs"    import os    os.makedirs(OUTPUT_DIR, exist_ok=True)    # Cache weights on Drive too, so a reconnect does not re-download them.    os.environ["HF_HOME"] = "/content/drive/MyDrive/kleos/hf_cache"else:    OUTPUT_DIR = "outputs"print("checkpoints ->", OUTPUT_DIR)

## 8. Dry runValidates the config, dataset and feasibility, writes a manifest, and stopsbefore loading any weights. Cheap insurance.

In [ ]:
!python scripts/train.py --config {CONFIG} --dataset {DATASET} --output-dir {OUTPUT_DIR} --dry-run

## 9. TrainThe pre-flight report prints GPU, VRAM, CUDA and library versions, the model andquantization settings, the LoRA configuration and a memory estimate. Training thenverifies that gradients actually reach the adapter before starting the real loop —a setup that trains nothing would otherwise still produce a plausible loss curve.

In [ ]:
!python scripts/train.py \    --config {CONFIG} \    --dataset {DATASET} \    --output-dir {OUTPUT_DIR}

## 10. Resume after a disconnectIf the runtime died, re-run sections 1-3 and 7, then this cell. `auto` finds thenewest **valid** checkpoint; a half-written checkpoint from an interrupted save isdetected and skipped rather than causing a confusing failure.

In [ ]:
!python scripts/train.py \    --config {CONFIG} \    --dataset {DATASET} \    --output-dir {OUTPUT_DIR} \    --resume-from-checkpoint auto

## 11. Inspect the run

In [ ]:
from kleos_models.experiments.registry import ExperimentRegistryregistry = ExperimentRegistry(OUTPUT_DIR)print(registry.render())latest = registry.scan()[0] if registry.scan() else Noneif latest:    print()    print(latest.manifest.summary())    ADAPTER = str(latest.directory / "adapter")    print()    print("adapter:", ADAPTER)

The manifest ties this artifact to model + revision + dataset version + datasethash + config hash + seed + git commit + environment. Check `adjustments` — if therun was automatically reshaped to fit memory, it is recorded there, and it affectscomparability.## 12. Optional: publish the adapterNothing is published automatically. This uploads adapter weights and a generatedmodel card, and refuses to upload training data, `.env`, or anything matching theprivate-data scanner.

In [ ]:
# Review what would be uploaded first.# !python scripts/publish_adapter.py --adapter {ADAPTER} --repo-id YOUR_USERNAME/kleos-qwen3-8b --dry-run# Then publish:# !python scripts/publish_adapter.py --adapter {ADAPTER} --repo-id YOUR_USERNAME/kleos-qwen3-8b

## Next`03_evaluate.ipynb` — compare this adapter against the base model.**A training loss curve is not a result.** Whether this adapter is better isdecided by evaluation on a held-out split, not by the loss reaching a low number.